In [0]:
# ============================================================
# 03_gold_aggregations
# Healthcare Data Warehouse — Medallion Architecture
# Layer: Gold (Star Schema KPI Aggregations)
# Description: Reads Silver Delta tables, builds Star Schema
#              with 3 dimensions and 3 fact tables answering
#              3 business questions:
#              1. Patient health & demographics
#              2. Claims & billing KPIs
#              3. Denial root cause analysis
# ============================================================
 
# Configuration 

storage_account_name = "adlshospitaldwh"
container_landing    = "landing"
container_bronze     = "bronze"
container_silver     = "silver"
container_gold       = "gold"
 
application_id = "ad3497a3-a8be-4251-8e01-4b7bc1c781b4"
tenant_id      = "2be11f7b-4e63-4e64-b8a4-992816fe3359"
client_secret  = "*******"
 
# Service Principal OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
LANDING_PATH = f"abfss://{container_landing}@{storage_account_name}.dfs.core.windows.net/"
BRONZE_PATH  = f"abfss://{container_bronze}@{storage_account_name}.dfs.core.windows.net/"
SILVER_PATH  = f"abfss://{container_silver}@{storage_account_name}.dfs.core.windows.net/"
GOLD_PATH    = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/"
 
print(" Configuration complete")
print(f"   Silver : {SILVER_PATH}")
print(f"   Gold   : {GOLD_PATH}")

 Configuration complete
   Silver : abfss://silver@adlshospitaldwh.dfs.core.windows.net/
   Gold   : abfss://gold@adlshospitaldwh.dfs.core.windows.net/


In [0]:
#  Create Silver Temp Views
try:
    silver_tables = [
        "patients", "encounters", "diagnoses", "claims_and_billing",
        "denials", "procedures", "medications", "providers", "lab_tests"
    ]
    for table in silver_tables:
        spark.read.format("delta").load(f"{SILVER_PATH}{table}").createOrReplaceTempView(table)
 
    print(" All silver views created — ready for SQL!")
 
except Exception as e:
    print(f" FAILED — silver views: {e}")
    raise
 
 


 All silver views created — ready for SQL!


In [0]:

#   dim_patient ────────────────────────
try:
    df_dim_patient = spark.sql("""
        SELECT
            patient_id,
            first_name,
            last_name,
            dob,
            age,
            CASE
                WHEN age < 18             THEN 'Under 18'
                WHEN age BETWEEN 18 AND 34 THEN '18-34'
                WHEN age BETWEEN 35 AND 49 THEN '35-49'
                WHEN age BETWEEN 50 AND 64 THEN '50-64'
                ELSE '65+'
            END AS age_group,
            gender,
            ethnicity,
            insurance_type,
            marital_status,
            city,
            state,
            zip,
            registration_date
        FROM patients
    """)
    df_dim_patient.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}dim_patient")
    print(f" dim_patient: {df_dim_patient.count():,} rows")
 
except Exception as e:
    print(f" FAILED — dim_patient: {e}")
    raise


 dim_patient: 60,000 rows


In [0]:
# dim_provider ─────────────────────────────────────
try:
    df_dim_provider = spark.sql("""
        SELECT
            provider_id,
            name,
            department,
            specialty,
            npi,
            inhouse,
            location,
            years_experience
        FROM providers
    """)
    df_dim_provider.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}dim_provider")
    print(f" dim_provider: {df_dim_provider.count():,} rows")
 
except Exception as e:
    print(f" FAILED — dim_provider: {e}")
    raise
 

 dim_provider: 1,491 rows


In [0]:
# dim_date ─────────────────────────────────────────
try:
    df_dim_date = spark.sql("""
        SELECT DISTINCT
            visit_date                          AS full_date,
            DAY(visit_date)                     AS day,
            MONTH(visit_date)                   AS month,
            DATE_FORMAT(visit_date, 'MMMM')     AS month_name,
            QUARTER(visit_date)                 AS quarter,
            YEAR(visit_date)                    AS year,
            DATE_FORMAT(visit_date, 'EEEE')     AS day_of_week,
            CASE
                WHEN DAYOFWEEK(visit_date) IN (1,7) THEN 'Yes'
                ELSE 'No'
            END AS is_weekend
        FROM encounters
        WHERE visit_date IS NOT NULL
    """)
    df_dim_date.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}dim_date")
    print(f" dim_date: {df_dim_date.count():,} rows")
 
except Exception as e:
    print(f" FAILED — dim_date: {e}")
    raise

 dim_date: 90 rows


In [0]:

 
# fact_encounters ──────────────────────────────────
try:
    df_fact_encounters = spark.sql("""
        SELECT
            e.encounter_id,
            e.patient_id,
            e.provider_id,
            e.visit_date,
            e.visit_type,
            e.department,
            e.admission_type,
            e.discharge_date,
            e.length_of_stay,
            e.status,
            e.readmitted_flag,
            e.diagnosis_code,
            CASE
                WHEN p.age < 18             THEN 'Under 18'
                WHEN p.age BETWEEN 18 AND 34 THEN '18-34'
                WHEN p.age BETWEEN 35 AND 49 THEN '35-49'
                WHEN p.age BETWEEN 50 AND 64 THEN '50-64'
                ELSE '65+'
            END AS age_group,
            p.gender,
            p.ethnicity,
            p.insurance_type,
            p.city,
            p.state,
            p.zip,
            YEAR(e.visit_date)                  AS visit_year,
            MONTH(e.visit_date)                 AS visit_month,
            DATE_FORMAT(e.visit_date, 'MMMM')   AS visit_month_name,
            QUARTER(e.visit_date)               AS visit_quarter
        FROM encounters e
        LEFT JOIN patients p ON e.patient_id = p.patient_id
    """)
    df_fact_encounters.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}fact_encounters")
    print(f" fact_encounters: {df_fact_encounters.count():,} rows")
 
except Exception as e:
    print(f" FAILED — fact_encounters: {e}")
    raise

 fact_encounters: 70,000 rows


In [0]:
# fact_claims ──────────────────────────────────────
try:
    df_fact_claims = spark.sql("""
        SELECT
            c.billing_id,
            c.patient_id,
            c.encounter_id,
            c.claim_id,
            c.insurance_provider,
            c.payment_method,
            c.claim_billing_date,
            c.billed_amount,
            c.paid_amount,
            c.billed_amount - c.paid_amount     AS unpaid_amount,
            c.claim_status,
            c.denial_reason,
            CASE
                WHEN p.age < 18             THEN 'Under 18'
                WHEN p.age BETWEEN 18 AND 34 THEN '18-34'
                WHEN p.age BETWEEN 35 AND 49 THEN '35-49'
                WHEN p.age BETWEEN 50 AND 64 THEN '50-64'
                ELSE '65+'
            END AS age_group,
            p.gender,
            p.ethnicity,
            p.city,
            p.state,
            p.zip,
            YEAR(c.claim_billing_date)              AS claim_year,
            MONTH(c.claim_billing_date)             AS claim_month,
            DATE_FORMAT(c.claim_billing_date,'MMMM') AS claim_month_name,
            QUARTER(c.claim_billing_date)           AS claim_quarter
        FROM claims_and_billing c
        LEFT JOIN patients p ON c.patient_id = p.patient_id
    """)
    df_fact_claims.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}fact_claims")
    print(f" fact_claims: {df_fact_claims.count():,} rows")
 
except Exception as e:
    print(f" FAILED — fact_claims: {e}")
    raise
 

 fact_claims: 70,000 rows


In [0]:
#fact_denials ─────────────────────────────────────
try:
    df_fact_denials = spark.sql("""
        SELECT
            d.denial_id,
            d.claim_id,
            d.denial_reason_code,
            d.denial_reason_description,
            d.denied_amount,
            d.denial_date,
            d.appeal_filed,
            d.appeal_status,
            d.appeal_resolution_date,
            d.final_outcome,
            c.patient_id,
            c.insurance_provider,
            c.billed_amount,
            c.paid_amount,
            CASE
                WHEN p.age < 18             THEN 'Under 18'
                WHEN p.age BETWEEN 18 AND 34 THEN '18-34'
                WHEN p.age BETWEEN 35 AND 49 THEN '35-49'
                WHEN p.age BETWEEN 50 AND 64 THEN '50-64'
                ELSE '65+'
            END AS age_group,
            p.gender,
            p.ethnicity,
            p.city,
            p.state,
            p.zip,
            YEAR(d.denial_date)                 AS denial_year,
            MONTH(d.denial_date)                AS denial_month,
            DATE_FORMAT(d.denial_date, 'MMMM')  AS denial_month_name,
            QUARTER(d.denial_date)              AS denial_quarter
        FROM denials d
        LEFT JOIN claims_and_billing c ON d.claim_id   = c.claim_id
        LEFT JOIN patients           p ON c.patient_id = p.patient_id
    """)
    df_fact_denials.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}fact_denials")
    print(f" fact_denials: {df_fact_denials.count():,} rows")
 
except Exception as e:
    print(f" FAILED — fact_denials: {e}")
    raise
 

 fact_denials: 5,998 rows


In [0]:
# Gold Layer Verification ──────────────────────────
gold_tables = {
    "dim_patient":      60_000,
    "dim_provider":      1_491,
    "dim_date":             90,
    "fact_encounters":  70_000,
    "fact_claims":      70_000,
    "fact_denials":      5_998,
}
 
print("\n" + "=" * 50)
print("GOLD LAYER SUMMARY")
print("=" * 50)
for table, expected in gold_tables.items():
    try:
        df     = spark.read.format("delta").load(f"{GOLD_PATH}{table}")
        actual = df.count()
        status = "" if actual == expected else " COUNT MISMATCH"
        print(f"{status} {table}: {actual:,} rows | {len(df.columns)} cols | expected {expected:,}")
    except Exception as e:
        print(f" Could not read {table}: {e}")
print("=" * 50)
print(" Gold layer complete!")


GOLD LAYER SUMMARY
 dim_patient: 60,000 rows | 14 cols | expected 60,000
 dim_provider: 1,491 rows | 8 cols | expected 1,491
 dim_date: 90 rows | 8 cols | expected 90
 fact_encounters: 70,000 rows | 23 cols | expected 70,000
 fact_claims: 70,000 rows | 22 cols | expected 70,000
 fact_denials: 5,998 rows | 24 cols | expected 5,998
 Gold layer complete!
